# 00 - Main study: T1 and T1b (and the faithful replay of the original T2 run)

Run this notebook first, on a **GPU** runtime. It computes the RoBERTa section embeddings once and
caches them in the results folder; notebooks 01-09 reuse the cache and run on a CPU.


Re-runs the published pipeline (section-tag-corrected RoBERTa embeddings, logistic
regression tuned by Hunger Games Search against a fuzzy-OWA cross-validated objective)
on **every task the corpus supports**, with the controls a reviewer will ask for.

**Label semantics** (official README, github.com/navid-aub/LinkedIn-Dataset):

| code | meaning | n |
|---|---|---|
| 0  | legitimate profile (LLP) | 1,800 |
| 1  | manually created fake profile (FLP) | 600 |
| 10 | ChatGPT-generated profile, built from *legitimate* profiles' statistics | 600 |
| 11 | ChatGPT-generated profile, built from *fake* profiles' statistics | 600 |

Codes 10 and 11 are **both synthetic ChatGPT profiles**; neither is genuine.

**Tasks run here**

| task | negative | positive | purpose |
|---|---|---|---|
| T1 | 0 legitimate | 1 manual fake | Gulati et al. protocol (1,800 / 600, 70/30 -> 540/180 test) |
| T1b | 0 legitimate (600 sampled) | 1 manual fake | Ayoobi et al. protocol (balanced, 420/420 train, 180/180 test) |
| T2 | 0 + 1 human-made | 10 + 11 ChatGPT-generated | **what the original headline notebook actually computed** (`test >= 10`) |
| T3 | 0 legitimate | 1 + 10 + 11 any fake | "is this profile fake at all" (optional) |

**What is kept from the original notebook, verbatim:** the cleaning pipeline, the RoBERTa
section-tag embedding, the fuzzy-OWA cross-validated fitness with the gap penalty, the HGS
optimiser, and the logistic-regression search space.

**What is fixed or added**

1. Labels built explicitly per task from the raw code - no implicit `>= 10` rule.
2. Scaling fitted on the training split only (the original fitted a scaler on all 3,600 rows first).
3. Repeated stratified splits (several seeds) instead of one, with mean, SD and bootstrap CI.
4. Matched-budget **random search** over the same grid, given exactly as many unique evaluations as HGS used.
5. **Exhaustive search** over the 336 distinct models (the 1,134-point grid collapses to 336 once invalid solver pairs and unused `l1_ratio` values are merged).
6. Positive-class, macro **and** weighted metrics reported side by side, plus PR-AUC and MCC.
7. Ablations: the tag correction on/off, and the `Full Name` field in/out of the Intro text.
8. A **faithful-replay** mode that reproduces the original run exactly (T2, seed 42, original scaling) as a sanity check before anything new is trusted.

Set a GPU runtime. Run top to bottom.

## 0. Setup

In [ ]:
!pip -q install transformers scikit-learn
import os
try:                                   # Google Colab: data and results on Google Drive
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except ImportError:                    # local run from the notebooks/ folder
    IN_COLAB = False
DATA_DIR = os.environ.get("LINKEDIN_DATA_DIR",
    "/content/drive/MyDrive/fake_profile_paper" if IN_COLAB else os.path.abspath("../data"))
OUT_DIR = os.environ.get("LINKEDIN_OUT_DIR",
    "/content/drive/MyDrive/fake_profile_paper/linkedin_repro" if IN_COLAB else os.path.abspath("../results/run"))
print("data:", DATA_DIR, "| results:", OUT_DIR)

import os, ast, time, logging
import numpy as np, pandas as pd, torch, joblib
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
RANDOM_STATE = 42
os.makedirs('embeddings_output', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

# ==== CELL ====
import json, math, random, itertools, warnings
from sklearn.metrics import average_precision_score, matthews_corrcoef, balanced_accuracy_score
warnings.filterwarnings("ignore")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- experiment switches ----------------
SEEDS            = [42, 7, 123]      # 3 seeds ~ comfortable in one session; add 2024, 99 for 5
RUN_TASKS        = ["T1", "T1b"]       # T2 and T3 run in notebooks 01-04
RUN_RANDOM       = True              # matched-budget random search
RUN_GRID         = True              # exhaustive 336-model search (first seed only; slow)
RUN_ABLATIONS    = False             # ablations run in notebook 05
FAITHFUL_REPLAY  = True              # reproduce the original T2 run before anything else
HGS_KW = dict(pop_size=12, max_iter=18, early_stop_patience=4)   # as in the original run

## 1. Load the corpus and recover the raw 4-way code

Works with either `LinkedIn_Dataset.csv` (label column `test`) or `LinkedIn_Dataset.pcl`
(label column `Label`). The counts are checked against the README before anything runs.

In [ ]:
def load_raw():
    csv = os.path.join(DATA_DIR, "LinkedIn_Dataset.csv")
    pcl = os.path.join(DATA_DIR, "LinkedIn_Dataset.pcl")
    if os.path.exists(csv):
        df = pd.read_csv(csv); col = "test" if "test" in df.columns else "Label"
    elif os.path.exists(pcl):
        df = pd.read_pickle(pcl); col = "Label" if "Label" in df.columns else "test"
    else:
        raise FileNotFoundError("put LinkedIn_Dataset.csv or .pcl in " + DATA_DIR)
    code = pd.to_numeric(df[col], errors="coerce").astype(int).values
    return df.reset_index(drop=True), code

import hashlib
OFFICIAL_SHA256 = "6d67c8015fceee06226491a8b0683bfc01e220416fed99e7f2d91aca27ba2e1a"   # LinkedIn_Dataset.pcl, github.com/navid-aub/LinkedIn-Dataset
_p = os.path.join(DATA_DIR, "LinkedIn_Dataset.pcl")
if os.path.exists(os.path.join(DATA_DIR, "LinkedIn_Dataset.csv")):
    print("note: LinkedIn_Dataset.csv is present and is used instead of the official .pcl file")
elif os.path.exists(_p):
    _h = hashlib.sha256(open(_p, "rb").read()).hexdigest()
    print("dataset checksum:", "matches the official release" if _h == OFFICIAL_SHA256
          else "WARNING - differs from the official release (%s)" % _h)

dataset, RAW_CODE = load_raw()
counts = pd.Series(RAW_CODE).value_counts().sort_index().to_dict()
print("rows:", len(dataset), "| raw code counts:", counts)
assert counts == {0: 1800, 1: 600, 10: 600, 11: 600}, "counts differ from the README - check the file"
print("OK: matches the README (0=LLP, 1=FLP, 10/11=ChatGPT-generated)")

## 2. Task definitions

In [ ]:
TASKS = {
  "T1":  dict(neg=[0],    pos=[1],        balance_neg=None, name="legitimate vs manual fake (Gulati protocol)"),
  "T1b": dict(neg=[0],    pos=[1],        balance_neg=600,  name="legitimate vs manual fake (Ayoobi, balanced)"),
  "T2":  dict(neg=[0, 1], pos=[10, 11],   balance_neg=None, name="human-made vs ChatGPT-generated (original headline)"),
  "T3":  dict(neg=[0],    pos=[1, 10, 11],balance_neg=None, name="legitimate vs any fake"),
}
def task_index(task, seed):
    t = TASKS[task]
    neg = np.where(np.isin(RAW_CODE, t["neg"]))[0]
    pos = np.where(np.isin(RAW_CODE, t["pos"]))[0]
    if t["balance_neg"]:
        neg = np.random.default_rng(seed).choice(neg, t["balance_neg"], replace=False)
    idx = np.sort(np.r_[neg, pos])
    y = np.isin(RAW_CODE[idx], t["pos"]).astype(int)
    return idx, y
for k in RUN_TASKS:
    idx, y = task_index(k, 42)
    print("%-4s %-52s n=%-5d positives=%-5d (%.1f%%)" % (k, TASKS[k]["name"], len(y), y.sum(), 100*y.mean()))

## 3. Cleaning - original code, verbatim

Label-independent string cleaning; it fits no statistics, so running it on all rows leaks nothing.

In [ ]:
REQUIRED_COLUMNS = [
    'Intro','Full Name','Workplace','Location','Connections','Photo','Followers','About',
    'Experiences','Number of Experiences','Educations','Number of Educations','Licenses',
    'Number of Licenses','Volunteering','Number of Volunteering','Skills','Number of Skills',
    'Recommendations','Number of Recommendations','Projects','Number of Projects',
    'Publications','Number of Publications','Courses','Number of Courses','Honors',
    'Number of Honors','Scores','Number of Scores','Languages','Number of Languages',
    'Organizations','Number of Organizations','Interests','Number of Interests',
    'Activities','Number of Activities','Label']

def get_dtype(col):
    if col == 'Label': return 'Int64'
    if 'Number of' in col: return 'Int64'
    return 'string'

def parse_chatgpt_content(content):
    parts = content.split('#@')
    if len(parts) >= 4:
        return {'Full Name': parts[0].strip(),
                'Workplace': f"{parts[1].strip()} at {parts[2].strip()}",
                'Location':  parts[3].strip(),
                'Connections': '500+', 'Photo': 'No', 'Followers': ''}
    return {k: '' for k in ['Full Name','Workplace','Location','Connections','Photo','Followers']}

def clean_chatgpt_field(field, name):
    try:
        if isinstance(field, str):
            field = ast.literal_eval(field)
        if isinstance(field, dict) and 'chatGPT' in field:
            content = field['chatGPT']
            first = content.split('\n')[0]
            if first[:2] in ('1.','2.','3.','4.','5.'):
                first = first.split('.', 1)[1].strip()
            if name == 'Intro': return parse_chatgpt_content(first)
            if name == 'Full Name': return first.split('#@')[0].strip()
            if name == 'Workplace':
                p = first.split('#@')
                return f"{p[1].strip()} at {p[2].strip()}" if len(p) > 2 else ''
            if name == 'Location':
                p = first.split('#@')
                return p[3].strip() if len(p) > 3 else ''
            return first.replace('#@', ' ')
        if isinstance(field, dict): return field
    except Exception:
        return parse_chatgpt_content('') if name == 'Intro' else ''
    return field if field is not None else ''

def safe_eval(v):
    if isinstance(v, str):
        try: return ast.literal_eval(v)
        except Exception: return v
    return v

def to_float(v):
    try: return float(v)
    except Exception: return 0.0

NUMERICAL_FIELDS = ['Number of Experiences','Number of Educations','Number of Licenses',
                    'Number of Volunteering','Number of Skills','Number of Recommendations',
                    'Number of Projects','Number of Publications','Number of Courses',
                    'Number of Honors','Number of Scores','Number of Languages',
                    'Number of Organizations','Number of Interests','Number of Activities']
NUMERICAL_FIELD_NAMES = NUMERICAL_FIELDS

def clean_dataset(df):
    cleaned = pd.DataFrame(columns=REQUIRED_COLUMNS)
    for c in REQUIRED_COLUMNS:
        cleaned[c] = pd.Series(dtype=get_dtype(c))
    for c in df.columns:
        if c in REQUIRED_COLUMNS and c != 'Label':
            cleaned[c] = df[c]
    if 'Label' in df.columns:
        cleaned['Label'] = pd.to_numeric(df['Label'], errors='coerce').fillna(0).astype(int)
    else:
        # Définition de label : test ≥ 10 → 1 sinon 0 (cohérent avec notebook CatBoost LinkedIn)
        if 'test' in df.columns:
            cleaned['Label'] = df['test'].apply(lambda v: 1 if to_float(v) >= 10 else 0).astype(int)
        else:
            cleaned['Label'] = 0
    for c in ('Intro','Full Name','Workplace','Location'):
        if c in cleaned.columns:
            cleaned[c] = cleaned[c].apply(lambda v: clean_chatgpt_field(v, c))
    # Conditional imputation depuis Intro
    intros = cleaned['Intro'].apply(lambda v: v if isinstance(v, dict) else parse_chatgpt_content(''))
    for k in ('Full Name','Workplace','Location','Connections','Photo','Followers'):
        mask = cleaned[k].astype(str).str.strip().eq('') | cleaned[k].isna()
        cleaned.loc[mask, k] = intros[mask].apply(lambda d: d.get(k, ''))
    cleaned = cleaned.fillna('')
    return cleaned


cleaned_df = clean_dataset(dataset)
print("Cleaned:", cleaned_df.shape)

## 4. Embeddings - original `embed_text`, cached per section

The original computes `(e(intro)-e("Introduction") + e(about)-e("About") + e(exp)-e("Experiences"))/3`.
Content embeddings are cached separately so the tag-corrected and plain variants come free, and the
Intro is embedded twice: with `Full Name` (as the original did) and without it.

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta   = RobertaModel.from_pretrained('roberta-base').to(device).eval()

@torch.no_grad()
def embed_text(text):
    if not text or not text.strip():
        return np.zeros(roberta.config.hidden_size, dtype=np.float32)
    inp = tokenizer(text, return_tensors='pt', padding=True,
                    truncation=True, max_length=512).to(device)
    out = roberta(**inp).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return out.astype(np.float32)

def section_text_embedding(content, tag):
    return embed_text(content) - embed_text(tag)


def section_texts(df, include_name=True):
    rows = []
    for i in range(len(df)):
        intro = safe_eval(df['Intro'].iloc[i]); about = safe_eval(df['About'].iloc[i])
        exps  = safe_eval(df['Experiences'].iloc[i])
        if isinstance(intro, dict):
            parts = ([intro.get('Full Name','')] if include_name else []) + \
                    [intro.get('Workplace',''), intro.get('Location','')]
            intro_text = " ".join(str(p) for p in parts)
        else:
            intro_text = str(intro)
        about_text = about.get('About','') if isinstance(about, dict) else str(about)
        exp_text = ""
        if isinstance(exps, dict): exps = exps.values()
        if isinstance(exps, (list, type({}.values()))):
            for e in exps:
                if isinstance(e, dict):
                    exp_text += " ".join(str(e.get(k,'')) for k in
                        ['Role','Workplace','Duration','Workplace Location','Description']) + " "
        rows.append((intro_text, about_text, exp_text))
    return rows

def cached_section_embeddings(include_name):
    path = os.path.join(OUT_DIR, "roberta_sections_name%d.npy" % int(include_name))
    if os.path.exists(path):
        return np.load(path)
    rows = section_texts(cleaned_df, include_name)
    E = np.zeros((len(rows), 3, roberta.config.hidden_size), dtype=np.float32)
    for i, (a, b, c) in enumerate(rows):
        E[i, 0], E[i, 1], E[i, 2] = embed_text(a), embed_text(b), embed_text(c)
        if (i+1) % 300 == 0: print("  embedded", i+1, "/", len(rows))
    np.save(path, E); return E

TAG_E = np.stack([embed_text("Introduction"), embed_text("About"), embed_text("Experiences")])
SEC_NAME   = cached_section_embeddings(include_name=True)    # original behaviour
SEC_NONAME = cached_section_embeddings(include_name=False)   # always cached, used by notebook 05

def text_features(sec, tag_correction=True):
    return ((sec - TAG_E[None]) if tag_correction else sec).mean(axis=1)

NUM_X = np.array([[to_float(cleaned_df[f].iloc[i]) for f in NUMERICAL_FIELDS]
                  for i in range(len(cleaned_df))], dtype=np.float32)
print("sections:", SEC_NAME.shape, "| numeric:", NUM_X.shape)

## 5. Fuzzy-OWA fitness and Hunger Games Search - original code, verbatim

In [ ]:
# =============================================================================
#  MODULE : Hunger Games Search (HGS) + Fuzzy OWA Aggregation Anti-Overfitting
#  ---------------------------------------------------------------------------
#  Référence HGS : Yang Y., Chen H., Heidari A.A., Gandomi A.H. (2021)
#                  "Hunger Games Search", ESWA 177, 114864.
#  Référence OWA  : Yager R.R. (1988) "On Ordered Weighted Averaging Aggregation
#                                      Operators in Multi-Criteria Decision Making"
#
#  STRATÉGIE ANTI-OVERFITTING :
#  ---------------------------------------------------------------------------
#  1)  Le test set n'est JAMAIS vu par le fitness (sacré jusqu'à l'évaluation
#      finale). Fitness = K-fold CV stratifiée sur (X_train, y_train).
#  2)  Agrégation FLOUE (OWA pessimiste) des K scores : on donne plus de poids
#      aux PIRES plis → pénalise la variance entre folds → robustesse.
#  3)  Fuzzy multi-metric : fusion F1 ⊕ AUC ⊕ Recall par OWA pondéré (chaque
#      métrique est min-max normalisée).
#  4)  Pénalité de généralisation : fitness final = OWA_cv - λ · (train - μ_cv)
#      → décourage les modèles qui sur-apprennent (gap train-CV élevé).
#  5)  Early-stopping HGS : si pas d'amélioration pendant `patience` itérations,
#      on arrête → moins d'évaluations, moins de risque de surajustement à un
#      pli particulier.
# =============================================================================
import math, random, time
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import f1_score, roc_auc_score, recall_score, accuracy_score


# -----------------------------------------------------------------------------
#  1. Fuzzy OWA (Ordered Weighted Average) — Yager 1988
# -----------------------------------------------------------------------------
def owa_weights(n, orness=0.30):
    """
    Génère des poids OWA pour un vecteur de taille n.
        orness ∈ [0,1] :
            • 0.0  → "AND" (min)              très pessimiste
            • 0.5  → moyenne arithmétique
            • 1.0  → "OR"  (max)              très optimiste
        On choisit orness = 0.30 (légèrement pessimiste) : on récompense
        les modèles dont les PIRES folds restent corrects.
    Implémentation : poids exponentiels biaisés vers la fin (queue basse).
    """
    if n <= 1:
        return np.ones(n) / max(n, 1)
    # Distribution exponentielle : alpha contrôle la pente
    # alpha = -ln(orness)  ⇒ pour orness < 0.5, poids concentrés sur les PETITES
    #                       valeurs (premiers éléments du vecteur trié ascending)
    #                       → moyenne pondérée tirée vers le bas = pessimiste.
    alpha = max(-math.log(max(orness, 1e-6)), 1e-6)
    idx = np.arange(n)
    # poids décroissants : w[0] (pire valeur) = max, w[n-1] (meilleure) = min
    w = np.exp(-alpha * idx / max(n - 1, 1))
    return w / w.sum()


def fuzzy_owa(scores, orness=0.30):
    """
    Agrégation OWA d'un vecteur de scores.
    1) Trier les scores en ordre croissant.
    2) Appliquer w_i tel que les premiers (pires) reçoivent plus de poids.
    """
    arr = np.asarray(scores, dtype=float).ravel()
    if arr.size == 0:
        return 0.0
    arr = np.sort(arr)                          # ordre croissant
    w   = owa_weights(arr.size, orness=orness)  # poids biaisés bas
    return float(np.dot(w, arr))


def fuzzy_multi_metric(y_true, y_pred, y_proba=None, orness=0.30):
    """
    Combine F1, Recall et AUC (si y_proba) en un score flou unique via OWA.
    Chaque métrique ∈ [0,1] → comparable.
    """
    binary = len(np.unique(y_true)) == 2
    avg = "binary" if binary else "macro"
    f1  = f1_score(y_true, y_pred, average=avg, zero_division=0)
    rec = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics = [f1, rec]
    if y_proba is not None and binary:
        try:
            auc = roc_auc_score(y_true, y_proba)
            metrics.append(auc)
        except Exception:
            pass
    return fuzzy_owa(metrics, orness=orness)


# -----------------------------------------------------------------------------
#  2. Fitness K-fold CV avec fuzzy aggregation + pénalité de gap
# -----------------------------------------------------------------------------
def make_cv_fitness(estimator_factory, X_train, y_train,
                    n_splits=5, seed=42, orness=0.30,
                    gap_penalty=0.50, use_proba=True):
    """
    Renvoie une fonction de fitness `fit(params) -> float` agrégée via OWA flou.

        estimator_factory : callable(params:dict) -> sklearn-compatible model
        n_splits          : nb de folds (5 par défaut)
        orness            : ∈ [0,1], <0.5 = pessimiste, pénalise variance
        gap_penalty       : λ, pénalité du gap (train - μ_cv) ∈ [0,1]
        use_proba         : True si l'estimator expose predict_proba (AUC)

    Le SCORE final renvoyé :
        score = OWA_cv  -  λ · max(0, train_full - mean_cv)
    où :
        OWA_cv     = agrégation floue OWA des K scores fuzzy_multi_metric
        train_full = score fuzzy_multi_metric sur l'ENSEMBLE train re-fit
        gap        = (train_full - mean_cv)  → quantifie l'overfitting
    """
    X_train = np.asarray(X_train) if not hasattr(X_train, "toarray") else X_train
    y_train = np.asarray(y_train)

    def _safe_score(est, X_tr, y_tr, X_va, y_va):
        try:
            est.fit(X_tr, y_tr)
            y_pred = est.predict(X_va)
            y_proba = None
            if use_proba and hasattr(est, "predict_proba"):
                try:
                    y_proba = est.predict_proba(X_va)[:, 1]
                except Exception:
                    y_proba = None
            return fuzzy_multi_metric(y_va, y_pred, y_proba, orness=orness)
        except Exception:
            return 0.0

    def fitness(params):
        # --- K-fold CV stratifiée sur X_train UNIQUEMENT (jamais X_test) ---
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        fold_scores = []
        try:
            est_template = estimator_factory(params)
        except Exception:
            return 0.0

        for tr_idx, va_idx in skf.split(np.zeros(len(y_train)), y_train):
            # sparse-safe indexing
            if hasattr(X_train, "tocsr"):
                Xtr_f, Xva_f = X_train.tocsr()[tr_idx], X_train.tocsr()[va_idx]
            else:
                Xtr_f, Xva_f = X_train[tr_idx], X_train[va_idx]
            ytr_f, yva_f = y_train[tr_idx], y_train[va_idx]
            est = clone(est_template)
            fold_scores.append(_safe_score(est, Xtr_f, ytr_f, Xva_f, yva_f))

        owa_cv  = fuzzy_owa(fold_scores, orness=orness)
        mean_cv = float(np.mean(fold_scores))

        # --- Score train complet pour calcul de gap (overfitting) ---
        est_full = clone(est_template)
        try:
            est_full.fit(X_train, y_train)
            y_pred_tr = est_full.predict(X_train)
            y_proba_tr = None
            if use_proba and hasattr(est_full, "predict_proba"):
                try:
                    y_proba_tr = est_full.predict_proba(X_train)[:, 1]
                except Exception:
                    y_proba_tr = None
            train_full = fuzzy_multi_metric(y_train, y_pred_tr, y_proba_tr, orness=orness)
        except Exception:
            train_full = mean_cv

        gap     = max(0.0, train_full - mean_cv)
        penalty = gap_penalty * gap
        return owa_cv - penalty

    return fitness


# -----------------------------------------------------------------------------
#  3. HGS Optimizer (avec early-stopping pour limiter le risque d'overfit)
# -----------------------------------------------------------------------------
class HGSOptimizer:
    """
    Hunger Games Search — Yang et al. 2021.
    Compatible recherche discrète : search_space = {param: [v1, v2, ...]}.

    Améliorations anti-overfitting :
        • early_stop_patience : arrête HGS si pas d'amélioration sur N itérations
        • Le fitness est typiquement obtenu via make_cv_fitness (fuzzy OWA + gap).
    """
    def __init__(self, search_space, fitness_fn,
                 pop_size=8, max_iter=15,
                 l_h=1e-2, LH=1e4,
                 seed=42, verbose=True,
                 early_stop_patience=4):
        self.space   = search_space
        self.keys    = list(search_space.keys())
        self.dims    = [len(search_space[k]) for k in self.keys]
        self.D       = len(self.keys)
        self.fit     = fitness_fn
        self.N       = pop_size
        self.T       = max_iter
        self.l_h     = l_h
        self.LH      = LH
        self.verbose = verbose
        self.patience = early_stop_patience
        random.seed(seed); np.random.seed(seed)
        self.history = []
        self.best_params_ = None
        self.best_score_  = -np.inf

    def _decode(self, x):
        params = {}
        for i, k in enumerate(self.keys):
            idx = int(min(self.dims[i] - 1, max(0, math.floor(x[i] * self.dims[i]))))
            params[k] = self.space[k][idx]
        return params

    def _evaluate(self, x, cache):
        params = self._decode(x)
        key = tuple(sorted(params.items(), key=lambda kv: kv[0]))
        if key in cache:
            return cache[key], params
        s = self.fit(params)
        cache[key] = s
        return s, params

    def optimize(self):
        X = np.random.rand(self.N, self.D)
        F = np.full(self.N, -np.inf)
        Hunger = np.zeros(self.N)
        cache = {}

        for i in range(self.N):
            F[i], _ = self._evaluate(X[i], cache)

        best_idx = int(np.argmax(F))
        X_best   = X[best_idx].copy()
        F_best   = F[best_idx]
        self.best_params_ = self._decode(X_best)
        self.best_score_  = F_best
        self.history.append(F_best)

        if self.verbose:
            print(f"[HGS] init  best fuzzy-score={F_best:.4f}  params={self.best_params_}")

        no_improve = 0
        for t in range(self.T):
            E_t = 2 * (1 - t / max(1, self.T))
            best_fit  = F.max();  worst_fit = F.min()

            for i in range(self.N):
                if abs(best_fit - worst_fit) < 1e-12:
                    TH = 0
                else:
                    TH = (F[i] - best_fit) / (best_fit - worst_fit + 1e-12) * \
                         random.random() * 2 * (1 - t / max(1, self.T))
                hunger_i = TH if abs(TH) > self.l_h else self.l_h
                Hunger[i] += min(self.LH, abs(hunger_i))

            S_hunger = Hunger.sum() + 1e-12
            W1 = np.zeros(self.N); W2 = np.zeros(self.N)
            for i in range(self.N):
                if random.random() < (Hunger[i] / S_hunger):
                    W1[i] = Hunger[i] * self.N / S_hunger * random.random()
                else:
                    W1[i] = 1.0
                W2[i] = (1 - math.exp(-abs(Hunger[i] - Hunger[best_idx]))) * \
                        random.random() * 2

            improved_iter = False
            for i in range(self.N):
                r1, r2 = random.random(), random.random()
                a = 2 * E_t * r1 - E_t
                R = a
                E = math.copysign(1, random.random() - 0.5) * \
                    (math.exp(-abs(F[i] - F_best) / (abs(F_best) + 1e-12)))

                levy_step = 0.01 * np.random.standard_cauchy(self.D)
                if r1 < self.l_h:
                    X_new = np.random.rand(self.D)
                elif r2 > E:
                    X_new = W1[i] * X_best + R * W2[i] * abs(X_best - X[i])
                else:
                    X_new = W1[i] * X_best - R * W2[i] * abs(X_best - X[i]) + levy_step
                X_new = np.clip(X_new, 0.0, 1.0)

                F_new, params_new = self._evaluate(X_new, cache)
                if F_new > F[i]:
                    X[i] = X_new
                    F[i] = F_new
                    if F_new > F_best:
                        F_best = F_new
                        X_best = X_new.copy()
                        best_idx = i
                        self.best_params_ = params_new
                        self.best_score_  = F_best
                        improved_iter = True
                        if self.verbose:
                            print(f"[HGS] t={t+1:02d}  ↑  fuzzy-score={F_best:.4f}  {params_new}")

            self.history.append(F_best)
            no_improve = 0 if improved_iter else (no_improve + 1)
            if no_improve >= self.patience:
                if self.verbose:
                    print(f"[HGS] early-stop @ t={t+1} (no improvement for {self.patience} iters)")
                break

        if self.verbose:
            print(f"[HGS] DONE   best fuzzy-score={self.best_score_:.4f}  "
                  f"params={self.best_params_}")
        return self.best_params_, self.best_score_, self.history


def hgs_search(search_space, fitness_fn,
               pop_size=8, max_iter=15, seed=42, verbose=True,
               early_stop_patience=4):
    """API simple : best_params, best_score, history = hgs_search(...)"""
    opt = HGSOptimizer(search_space, fitness_fn,
                       pop_size=pop_size, max_iter=max_iter,
                       seed=seed, verbose=verbose,
                       early_stop_patience=early_stop_patience)
    t0 = time.time()
    best_params, best_score, history = opt.optimize()
    print(f"[HGS] elapsed = {time.time()-t0:.1f}s")
    return best_params, best_score, history

## 6. Search space - original code, verbatim, plus the distinct-model enumeration

In [ ]:
LR_SPACE = {
    "C":            [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0],   # plafond abaissé
    "penalty":      ["l1", "l2", "elasticnet"],
    "solver":       ["liblinear", "saga", "lbfgs"],
    "class_weight": [None, "balanced"],
    "max_iter":     [500, 1000, 2000],
    "l1_ratio":     [0.1, 0.5, 0.9],
}

VALID_COMBOS = {
    ("l1", "liblinear"), ("l1", "saga"),
    ("l2", "liblinear"), ("l2", "saga"), ("l2", "lbfgs"),
    ("elasticnet", "saga"),
}

def build_estimator(params):
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS:
        # combinaison invalide : on retombe sur (l2, lbfgs)
        p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet":
        p.pop("l1_ratio", None)
    if p["solver"] == "liblinear":
        return LogisticRegression(**p, random_state=42, n_jobs=1)
    return LogisticRegression(**p, random_state=42, n_jobs=-1)


def canonical(params):
    # the model build_estimator() actually fits - used to merge duplicate grid points
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS: p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet": p["l1_ratio"] = None
    return tuple(sorted(p.items(), key=lambda kv: kv[0]))

DISTINCT = {}
for combo in itertools.product(*LR_SPACE.values()):
    p = dict(zip(LR_SPACE.keys(), combo)); DISTINCT.setdefault(canonical(p), p)
print("nominal grid:", int(np.prod([len(v) for v in LR_SPACE.values()])),
      "| distinct models:", len(DISTINCT))

## 7. Counting wrapper, random search, exhaustive search, metrics

In [ ]:
class Counted:
    # wraps a fitness so the number of DISTINCT models evaluated is known exactly
    def __init__(self, fn): self.fn, self.seen = fn, {}
    def __call__(self, params):
        k = canonical(params)
        if k not in self.seen: self.seen[k] = self.fn(params)
        return self.seen[k]
    @property
    def n(self): return len(self.seen)

def random_search(fitness, budget, seed):
    keys = list(DISTINCT.keys()); rng = random.Random(seed); rng.shuffle(keys)
    best_p, best_s = None, -1e9
    for k in keys[:budget]:
        s = fitness(DISTINCT[k])
        if s > best_s: best_p, best_s = DISTINCT[k], s
    return best_p, best_s

def grid_search(fitness):
    best_p, best_s = None, -1e9
    for i, p in enumerate(DISTINCT.values()):
        s = fitness(p)
        if s > best_s: best_p, best_s = p, s
        if (i+1) % 50 == 0: print("    grid", i+1, "/", len(DISTINCT), "best", round(best_s, 4))
    return best_p, best_s

def evaluate(y, proba, thr=0.5):
    yp = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()
    return dict(
        accuracy=accuracy_score(y, yp),
        precision_pos=precision_score(y, yp, zero_division=0),
        recall_pos=recall_score(y, yp, zero_division=0),
        f1_pos=f1_score(y, yp, zero_division=0),
        f1_macro=f1_score(y, yp, average="macro", zero_division=0),
        precision_w=precision_score(y, yp, average="weighted", zero_division=0),
        recall_w=recall_score(y, yp, average="weighted", zero_division=0),
        f1_w=f1_score(y, yp, average="weighted", zero_division=0),
        roc_auc=roc_auc_score(y, proba), pr_auc=average_precision_score(y, proba),
        mcc=matthews_corrcoef(y, yp), bal_acc=balanced_accuracy_score(y, yp),
        TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), n_test=int(len(y)))

## 8. Faithful replay of the original headline run (sanity check)

Same task (T2), seed 42, and the original **full-data** numeric scaling. If this does not
reproduce accuracy 0.9954 / F1 0.9931, stop: something in the environment differs and nothing
below can be compared with the article.

In [ ]:
if FAITHFUL_REPLAY:
    idx, y = task_index("T2", 42)
    Xt  = text_features(SEC_NAME[idx], True)
    Xn  = StandardScaler().fit_transform(NUM_X)[idx]          # original: fitted on all rows
    X   = np.hstack([Xt, Xn]).astype(np.float32)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
    sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    fit = Counted(make_cv_fitness(build_estimator, Xtr, ytr, n_splits=5, seed=42,
                                  orness=0.30, gap_penalty=0.50, use_proba=True))
    bp, bs, hist = hgs_search(LR_SPACE, fit, seed=42, verbose=False, **HGS_KW)
    m = build_estimator(bp).fit(Xtr, ytr)
    r = evaluate(yte, m.predict_proba(Xte)[:, 1])
    print("best params (as returned):", bp)
    print("best params (as fitted)  :", dict(canonical(bp)), "| distinct models evaluated:", fit.n)
    print("replay  acc=%.4f  P=%.4f  R=%.4f  F1=%.4f  AUC=%.4f" %
          (r["accuracy"], r["precision_pos"], r["recall_pos"], r["f1_pos"], r["roc_auc"]))
    print("article acc=0.9954  P=0.9890  R=0.9972  F1=0.9931  AUC=0.9999")

## 9. Main experiment: every task x seed x search strategy

Scaling is now fitted on the training split only. Each seed draws a new stratified 70/30 split
(and, for T1b, a new balanced subsample of legitimate profiles).

In [ ]:
def prepare(task, seed, sec=None, tag_correction=True):
    idx, y = task_index(task, seed)
    sec = SEC_NAME if sec is None else sec
    X = np.hstack([text_features(sec[idx], tag_correction), NUM_X[idx]]).astype(np.float32)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=seed, stratify=y)
    sc = StandardScaler().fit(Xtr)                              # training split only
    return sc.transform(Xtr), sc.transform(Xte), ytr, yte

def run_one(task, seed, strategy, sec=None, tag_correction=True, variant="main"):
    Xtr, Xte, ytr, yte = prepare(task, seed, sec, tag_correction)
    fit = Counted(make_cv_fitness(build_estimator, Xtr, ytr, n_splits=5, seed=seed,
                                  orness=0.30, gap_penalty=0.50, use_proba=True))
    t0 = time.time()
    if strategy == "HGS":
        bp, bs, _ = hgs_search(LR_SPACE, fit, seed=seed, verbose=False, **HGS_KW)
    elif strategy.startswith("Random"):
        bp, bs = random_search(fit, BUDGET[(task, seed)], seed)
    else:
        bp, bs = grid_search(fit)
    m = build_estimator(bp).fit(Xtr, ytr)
    r = evaluate(yte, m.predict_proba(Xte)[:, 1])
    r.update(task=task, seed=seed, strategy=strategy, variant=variant, cv_fitness=bs,
             n_models=fit.n, seconds=round(time.time()-t0, 1), params=json.dumps(bp),
             params_fitted=json.dumps(dict(canonical(bp))))   # what build_estimator actually fitted
    return r

RESULTS, BUDGET = [], {}
for task in RUN_TASKS:
    for si, seed in enumerate(SEEDS):
        r = run_one(task, seed, "HGS"); RESULTS.append(r); BUDGET[(task, seed)] = r["n_models"]
        print("%-4s seed %-4d HGS     models=%-3d F1+=%.4f acc=%.4f" % (task, seed, r["n_models"], r["f1_pos"], r["accuracy"]))
        if RUN_RANDOM:
            r = run_one(task, seed, "Random"); RESULTS.append(r)
            print("%-4s seed %-4d Random  models=%-3d F1+=%.4f acc=%.4f" % (task, seed, r["n_models"], r["f1_pos"], r["accuracy"]))
        if RUN_GRID and si == 0:
            r = run_one(task, seed, "Grid"); RESULTS.append(r)
            print("%-4s seed %-4d Grid    models=%-3d F1+=%.4f acc=%.4f" % (task, seed, r["n_models"], r["f1_pos"], r["accuracy"]))
        pd.DataFrame(RESULTS).to_csv(os.path.join(OUT_DIR, "results_raw.csv"), index=False)

## 10. Ablations (first seed): tag correction off, `Full Name` removed

In [ ]:
if RUN_ABLATIONS:
    seed = SEEDS[0]
    for task in RUN_TASKS:
        for variant, kw in [("no_tag_correction", dict(tag_correction=False)),
                            ("no_full_name",      dict(sec=SEC_NONAME))]:
            BUDGET[(task, seed)] = BUDGET.get((task, seed), 60)
            r = run_one(task, seed, "HGS", variant=variant, **kw); RESULTS.append(r)
            print("%-4s %-18s F1+=%.4f acc=%.4f" % (task, variant, r["f1_pos"], r["accuracy"]))
    pd.DataFrame(RESULTS).to_csv(os.path.join(OUT_DIR, "results_raw.csv"), index=False)

## 11. Summary tables

Mean and standard deviation across seeds, with a percentile bootstrap CI over seeds.
`*_w` columns are **weighted** averages, reported only for comparability with the thesis table and
with Gulati et al.; the positive-class columns are the ones to headline.

In [ ]:
R = pd.DataFrame(RESULTS)
keys = ["accuracy","precision_pos","recall_pos","f1_pos","f1_macro","f1_w","roc_auc","pr_auc","mcc","n_models"]
def boot(v, B=2000, rng=np.random.default_rng(0)):
    if len(v) < 2: return ""
    b = rng.choice(v, (B, len(v))).mean(1); return "[%.4f, %.4f]" % tuple(np.percentile(b, [2.5, 97.5]))
rows = []
for (task, strat, var), g in R.groupby(["task","strategy","variant"]):
    row = dict(task=task, strategy=strat, variant=var, seeds=len(g))
    for k in keys:
        row[k] = "%.4f +- %.4f" % (g[k].mean(), g[k].std(ddof=1) if len(g) > 1 else 0.0)
    row["f1_pos_CI"] = boot(g["f1_pos"].values)
    rows.append(row)
SUM = pd.DataFrame(rows); SUM.to_csv(os.path.join(OUT_DIR, "results_summary.csv"), index=False)
display(SUM)

print("\nHGS vs matched-budget random search, per task (positive-class F1, paired by seed):")
for task in RUN_TASKS:
    h = R[(R.task==task)&(R.strategy=="HGS")&(R.variant=="main")].set_index("seed")["f1_pos"]
    r = R[(R.task==task)&(R.strategy=="Random")&(R.variant=="main")].set_index("seed")["f1_pos"]
    d = (h - r).dropna()
    if len(d): print("  %-4s mean diff %+.4f  (HGS better on %d/%d seeds)" % (task, d.mean(), (d>0).sum(), len(d)))

## 12. Literature comparison (only where the protocol matches)

- **T1** matches Gulati et al. Table 1 (1,260/420 train, 540/180 test). Their baseline F1 is
  96.39-97.08% (Fig. 2a; Flair+CatBoost 96.4, Flair+XGBoost 97.1). Their averaging is not stated.
- **T1b** matches Ayoobi et al. **Table 2** (420/420 train, 180/180 test; SSTE accuracy 94.17-96.33%).
  Their figures are averages over five classifiers, and their features are text only and exclude
  recommendations and activity, whereas this pipeline adds 15 counts including both.
- **T2** has no published counterpart: both papers label manual fakes as fake, so T2 is an
  authorship-detection task.
- **T3** matches Gulati et al.'s GPT-3.5 Retrain / GPT-3.5 Attack cell (1,260/420/840 train,
  540/180/360 test; F1 97.7-98.2) when the split is stratified on the raw code - see notebook 04.